In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Carrega e prepara
df = pd.read_csv('campeonato-brasileiro-full.csv')
df = df.rename(columns={'mandante_Estado': 'mandante_estado', 'visitante_Estado': 'visitante_estado'})
df['data'] = pd.to_datetime(df['data'], dayfirst=True)
df = df.sort_values('data').reset_index(drop=True)

# 2. Resultado
df['resultado'] = np.where(df['mandante_Placar'] > df['visitante_Placar'], 0,
                np.where(df['mandante_Placar'] < df['visitante_Placar'], 2, 1))

# 3. FEATURE ENGINEERING CORRETO (com print pra provar que tá funcionando)
teams = list(set(df['mandante']) | set(df['visitante']))
stats = {t: {'hp':0, 'hw':0, 'hgf':0, 'hga':0, 'ap':0, 'aw':0, 'agf':0, 'aga':0} for t in teams}

features = []
for i, row in df.iterrows():
    h, a = row['mandante'], row['visitante']
    
    # Calcula ANTES do jogo
    features.append({
        'h_win': stats[h]['hw'] / max(stats[h]['hp'], 1),
        'a_win': stats[a]['aw'] / max(stats[a]['ap'], 1),
        'h_gf': stats[h]['hgf'] / max(stats[h]['hp'], 1),
        'a_gf': stats[a]['agf'] / max(stats[a]['ap'], 1),
    })
    
    # Atualiza DEPOIS
    stats[h]['hp'] += 1; stats[a]['ap'] += 1
    stats[h]['hgf'] += row['mandante_Placar']; stats[h]['hga'] += row['visitante_Placar']
    stats[a]['agf'] += row['visitante_Placar']; stats[a]['aga'] += row['mandante_Placar']
    if row['mandante_Placar'] > row['visitante_Placar']: stats[h]['hw'] += 1
    elif row['mandante_Placar'] < row['visitante_Placar']: stats[a]['aw'] += 1

    # PRINT DE DEBUG (só nos primeiros 5 e últimos 5 jogos)
    if i < 5 or i > len(df)-6:
        print(f"Jogo {i}: {h} {row['mandante_Placar']}x{row['visitante_Placar']} {a} → h_win_rate: {features[-1]['h_win']:.3f}")
# Junta
df = pd.concat([df, pd.DataFrame(features)], axis=1)

# Substitui zero por média
for col in ['h_win', 'a_win', 'h_gf', 'a_gf']:
    df[col] = df[col].replace(0, df[col].mean())

# Split
treino = df[df['data'].dt.year <= 2019]
teste = df[df['data'].dt.year >= 2020]

# Modelo SIMPLES com as 4 features que funcionam
X_train = treino[['h_win', 'a_win', 'h_gf', 'a_gf']]
X_test = teste[['h_win', 'a_win', 'h_gf', 'a_gf']]
y_train = treino['resultado']
y_test = teste['resultado']

model = RandomForestClassifier(n_estimators=500, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print(f"\n=== RESULTADO FINAL ===")
print(f"Acurácia: {accuracy_score(y_test, pred):.4f}")
print(f"Baseline: {(y_test == 0).mean():.4f}")

Jogo 0: Guarani 4x2 Vasco → h_win_rate: 0.000
Jogo 1: Athletico-PR 2x0 Gremio → h_win_rate: 0.000
Jogo 2: Flamengo 1x1 Coritiba → h_win_rate: 0.000
Jogo 3: Goias 2x2 Paysandu → h_win_rate: 0.000
Jogo 4: Internacional 1x1 Ponte Preta → h_win_rate: 0.000
Jogo 8780: Flamengo 2x2 Vitoria → h_win_rate: 0.552
Jogo 8781: Bahia 2x0 Atletico-GO → h_win_rate: 0.424
Jogo 8782: Juventude 0x1 Cruzeiro → h_win_rate: 0.422
Jogo 8783: Atletico-MG 1x0 Athletico-PR → h_win_rate: 0.549
Jogo 8784: Gremio 0x3 Corinthians → h_win_rate: 0.587

=== RESULTADO FINAL ===
Acurácia: 0.4416
Baseline: 0.4584


In [5]:
df = pd.read_csv('campeonato-brasileiro-full.csv')
print("Primeiras 3 linhas do seu arquivo:")
print(df.head(3)[['data', 'mandante', 'visitante', 'mandante_Placar', 'visitante_Placar']])
print("\nTotal de linhas:", len(df))
print("Colunas:", df.columns.tolist())

Primeiras 3 linhas do seu arquivo:
         data      mandante visitante  mandante_Placar  visitante_Placar
0  29/03/2003       Guarani     Vasco                4                 2
1  29/03/2003  Athletico-PR    Gremio                2                 0
2  30/03/2003      Flamengo  Coritiba                1                 1

Total de linhas: 8785
Colunas: ['ID', 'rodata', 'data', 'hora', 'mandante', 'visitante', 'formacao_mandante', 'formacao_visitante', 'tecnico_mandante', 'tecnico_visitante', 'vencedor', 'arena', 'mandante_Placar', 'visitante_Placar', 'mandante_Estado', 'visitante_Estado']
